In [1]:
import tensorflow as tf
import numpy as np


2023-08-10 20:29:05.173382: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-08-10 20:29:06.379567: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
input_file = 'input.wav'
output_file = 'output.wav'

input_wav = tf.io.read_file(input_file)
output_wav = tf.io.read_file(output_file)

2023-08-10 20:29:07.658913: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-08-10 20:29:07.718028: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-08-10 20:29:07.718107: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-08-10 20:29:07.750184: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:982] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-08-10 20:29:07.750302: I tensorflow/compile

In [3]:
channels = 1

input_lookback = 100

In [4]:
input_tensor, sample_rate = tf.audio.decode_wav(input_wav)
output_tensor, sample_rate = tf.audio.decode_wav(output_wav)

In [5]:
output_dataset = tf.data.Dataset.from_tensor_slices(output_tensor[input_lookback:len(output_tensor)])

output_dataset.element_spec


TensorSpec(shape=(1,), dtype=tf.float32, name=None)

In [6]:
input_dataset = tf.data.Dataset.from_tensor_slices(input_tensor)
input_dataset.element_spec

TensorSpec(shape=(1,), dtype=tf.float32, name=None)

In [7]:
input_dataset = input_dataset.window(input_lookback, 1)

input_dataset.element_spec

DatasetSpec(TensorSpec(shape=(1,), dtype=tf.float32, name=None), TensorShape([]))

In [8]:
def sub_to_batch(sub):
  return sub.batch(input_lookback, drop_remainder=True)

input_dataset = input_dataset.flat_map(sub_to_batch)

In [9]:
combined_dataset = tf.data.Dataset.zip((input_dataset, output_dataset))

In [10]:
training_dataset = combined_dataset.skip(44100 * 20).take(100)
training_dataset = training_dataset.shuffle(100).batch(32)
validation_dataset = combined_dataset.skip(44100 * 40).take(441).batch(32)
test_dataset = combined_dataset.skip(44100 * 60).take(44100 * 10)

In [11]:
model = tf.keras.Sequential([
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(1)
])

In [12]:
model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])

In [13]:
list(training_dataset.as_numpy_iterator())

2023-08-10 20:29:10.545218: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_8' with dtype float and shape [15523100,1]
	 [[{{node Placeholder/_8}}]]
2023-08-10 20:29:10.545810: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_8' with dtype float and shape [15523100,1]
	 [[{{node Placeholder/_8}}]]
2023-08-10 20:31:53.883783: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 1 of 100
2023-08-10 20:31:53.884071: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:392] Filling up shuffle buffer (this may take a while): 2 of 100
2023-08-10

[(array([[[ 0.0000000e+00],
          [ 6.1035156e-05],
          [ 9.1552734e-05],
          ...,
          [ 6.1035156e-05],
          [ 3.0517578e-05],
          [ 3.0517578e-05]],
  
         [[-6.1035156e-05],
          [-6.1035156e-05],
          [-6.1035156e-05],
          ...,
          [ 2.1362305e-04],
          [ 2.7465820e-04],
          [ 2.7465820e-04]],
  
         [[-9.1552734e-05],
          [-6.1035156e-05],
          [-6.1035156e-05],
          ...,
          [ 1.8310547e-04],
          [ 2.1362305e-04],
          [ 2.7465820e-04]],
  
         ...,
  
         [[-6.1035156e-05],
          [-6.1035156e-05],
          [-6.1035156e-05],
          ...,
          [ 1.5258789e-04],
          [ 1.5258789e-04],
          [ 1.5258789e-04]],
  
         [[-1.2207031e-04],
          [-1.2207031e-04],
          [-1.2207031e-04],
          ...,
          [ 2.4414062e-04],
          [ 2.4414062e-04],
          [ 2.4414062e-04]],
  
         [[-1.8310547e-04],
          [-1.831054

In [14]:
history = model.fit(
    training_dataset,
    validation_data = validation_dataset,
    epochs=3)

model.summary()

Epoch 1/3


In [ ]:
input_dataset.skip(44100 * 60)
prediction = model.predict(input_dataset.skip(44100 * 60).take(44100 * 10).batch(44100 * 10))

2023-08-10 20:13:45.590730: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype float and shape [15523200,1]
	 [[{{node Placeholder/_0}}]]
2023-08-10 20:13:45.591286: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype float and shape [15523200,1]
	 [[{{node Placeholder/_0}}]]


1/1 [==============================] - 369s 369s/step


In [ ]:
prediction_audio = tf.audio.encode_wav(prediction, 44100)
tf.io.write_file('prediction.wav', prediction_audio)